In [1]:
import pandas as pd
import numpy as np
import json
import os
import zipfile
import glob
import time
import xml.etree.ElementTree as ET

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Generating the common string for all filepaths

In [5]:
root_path = '/content/drive/MyDrive/Research stuff/Final Year Design Project/Datasets/Before pre-processing/CLEF_2016/'

negative_path = root_path + 'negative_examples_anonymous/'
positive_path = root_path + 'positive_examples_anonymous/'

negative_path_len = len(negative_path)
positive_path_len = len(positive_path)

print("length of the negative samples folder", negative_path_len)
print("length of the positive samples folder", positive_path_len)

length of the negative samples folder 133
length of the positive samples folder 133


Testing the data

In [6]:
test_data_path = positive_path + 'test_subject625.xml'

user_id = test_data_path [positive_path_len : -4]
print("User id is", user_id)

tree = ET.parse(test_data_path)
root = tree.getroot()

count = 0
for post in root.findall('./WRITING'):
    text = post.find('TEXT').text
    timestamp = post.find('DATE').text
    print(timestamp)
    break
    count += 1
#print(count)
ind = timestamp.find('2')
year = timestamp[ind : ind+4]
print(year)

User id is  test_subject625
 2015-06-30 21:01:11 
2015


Parsing the XML files belonging to the negative class and storing them in a list

In [23]:
final_negative_data = []
user_ids_in_negative_class = []

for name in glob.glob(negative_path + '*.xml'): 
    #extractng the user id from file name
    user_id = name[negative_path_len : -4]

    #appending it in the negative class label array
    user_label = {"user_id" : user_id, "y_label" : 0}
    user_ids_in_negative_class.append(user_label) 

    #parsing the XML tree
    tree = ET.parse(name)
    root = tree.getroot()

    for post in root.findall('./WRITING'):
        text = post.find('TEXT').text
        timestamp = post.find('DATE').text
        #print(timestamp, text)

        data = {'user_id': user_id, 'timestamp': None, 'text': None}
        data['timestamp'] = timestamp
        data['text'] = text

        final_negative_data.append(data)

Parsing the XML files belonging to the negative class and storing them in a list

In [24]:
final_positive_data = []
user_ids_in_positive_class = []

for name in glob.glob(positive_path + '*.xml'):
    #extractng the user id from file name
    user_id = name[positive_path_len : -4]

    #appending it in the negative class label array
    user_label = {"user_id" : user_id, "y_label" : 1}
    user_ids_in_positive_class.append(user_label)

    #parsing the XML tree
    tree = ET.parse(name)
    root = tree.getroot()

    for post in root.findall('./WRITING'):
        text = post.find('TEXT').text
        timestamp = post.find('DATE').text
        #print(timestamp, text)

        data = {'user_id': user_id, 'timestamp': None, 'text': None}
        data['timestamp'] = timestamp
        data['text'] = text

        final_positive_data.append(data)

Merging the lists

In [25]:
final_posts_data = final_negative_data + final_positive_data
print("Total posts = ", len(final_posts_data))

final_y_labels = user_ids_in_negative_class + user_ids_in_positive_class
print("Total users = ", len(final_y_labels)) 

Total posts =  531453
Total users =  892


Dataframe for the user posts

In [26]:
final_posts_data_df = pd.DataFrame(final_posts_data)
final_posts_data_df.head()

,user_id,timestamp,text
0,train_subject770,2015-06-10 19:22:39,"I already try to talk to her, in my experienc..."
1,train_subject770,2015-06-10 08:55:10,"Thank you, I might bring this up when Infeel ..."
2,train_subject770,2015-06-10 00:39:45,That must be so tough on you :( \n\nIf I may ...
3,train_subject770,2015-06-09 15:21:05,"Hi,\n\nThank you for your response.\n\nI have..."
4,train_subject770,2015-06-09 12:14:54,"Hi Reddit,\n\nMy little sister (she's 20, but..."


In [27]:
final_posts_data_df.shape

(531453, 3)

In [28]:
#Saving to excel format
final_data_df.to_excel('/content/clef_posts_data.xlsx', sheet_name = 'Sheet 1', encoding = 'utf-8')

Dataframe for y_labels

In [29]:
final_y_labels_df = pd.DataFrame(final_y_labels)
final_y_labels_df.head()

,user_id,y_label
0,train_subject770,0
1,train_subject6796,0
2,train_subject1033,0
3,train_subject7819,0
4,train_subject5081,0


In [30]:
final_y_labels_df.shape

(892, 2)

In [31]:
#Saving to excel format
final_y_labels_df.to_excel('/content/clef_users_mapped_to_class.xlsx', sheet_name = 'Sheet 1', encoding = 'utf-8')